# Test Word Document

This notebook loads a Word document, converts it to a Python dictionary using the `parse_to_dict` function from `repo root > preprocessing > assessment_processor.py`, parses the resulting assessment dictionary with `AssessmentParser`, and then generates rules-based violations with `IUCNAssessmentReviewer`.

By default it uses the bundled sample Word document in this folder, but you can point it at your own `.docx` file in the path cell below.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import json
import sys

TEST_WORD_DOCUMENT_ROOT = Path.cwd().resolve()
PACKAGE_ROOT = TEST_WORD_DOCUMENT_ROOT.parent
REPO_ROOT = PACKAGE_ROOT.parent

assert PACKAGE_ROOT.name == "iucn_rules_checker", (
    f"Expected notebook folder parent to be iucn_rules_checker, got {PACKAGE_ROOT}"
)
PROCESSOR_SCRIPT = REPO_ROOT / "preprocessing" / "assessment_processor.py"

assert TEST_WORD_DOCUMENT_ROOT.exists(), f"Could not find {TEST_WORD_DOCUMENT_ROOT}"
assert PROCESSOR_SCRIPT.exists(), f"Could not find {PROCESSOR_SCRIPT}"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def fresh_project_imports(verbose: bool = True):
    importlib.invalidate_caches()
    stale_modules = [
        name for name in list(sys.modules)
        if name == "iucn_rules_checker" or name.startswith("iucn_rules_checker.")
    ]
    for module_name in sorted(stale_modules, key=len, reverse=True):
        del sys.modules[module_name]

    assessment_parser_module = importlib.import_module("iucn_rules_checker.assessment_parser")
    assessment_reviewer_module = importlib.import_module("iucn_rules_checker.assessment_reviewer")

    result = {
        "AssessmentParser": assessment_parser_module.AssessmentParser,
        "IUCNAssessmentReviewer": assessment_reviewer_module.IUCNAssessmentReviewer,
    }

    if verbose:
        print("Using parser from:", Path(assessment_parser_module.__file__).resolve())
        print("Using reviewer from:", Path(assessment_reviewer_module.__file__).resolve())

    return result


def load_parse_to_dict(verbose: bool = True):
    module_name = "preprocessing_assessment_processor_test_word_document"
    if module_name in sys.modules:
        del sys.modules[module_name]

    spec = importlib.util.spec_from_file_location(module_name, PROCESSOR_SCRIPT)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)

    if verbose:
        print("Loaded parse_to_dict from:", PROCESSOR_SCRIPT.resolve())

    return module.parse_to_dict


In [ ]:
# Set CUSTOM_DOCX_PATH to a string or Path for your own `.docx` file.
# The original sample document is not included in the repo for confidentiality reasons.
CUSTOM_DOCX_PATH = None

if CUSTOM_DOCX_PATH is None:
    raise ValueError("Set CUSTOM_DOCX_PATH to a local .docx assessment file before running this notebook.")

DOCX_PATH = Path(CUSTOM_DOCX_PATH).expanduser()
assert DOCX_PATH.exists(), f"Could not find {DOCX_PATH}"
print("DOCX path:", DOCX_PATH)


## Run the Word-document conversion

This step converts the Word document into the structured assessment dictionary using the `parse_to_dict` function from `repo root > preprocessing > assessment_processor.py`.


In [ ]:
parse_to_dict = load_parse_to_dict(verbose=False)
assessment = parse_to_dict(str(DOCX_PATH))

print(type(assessment).__name__)
print("Top-level keys:", list(assessment.keys()))
print("Title:", assessment.get("title"))
print("Block count:", len(assessment.get("blocks", [])))
print("Child count:", len(assessment.get("children", [])))


## Parse the assessment dictionary

This step runs `AssessmentParser.parse(...)` to build the flat `full_report` mapping consumed by the reviewer.


In [ ]:
project_modules = fresh_project_imports(verbose=False)
parser = project_modules["AssessmentParser"]()
full_report = parser.parse(assessment)

paragraph_entries = [key for key in full_report if "[paragraph " in key]
table_row_entries = [key for key in full_report if "[table " in key and "[row " in key]

print(type(full_report).__name__)
print("Parsed entries:", len(full_report))
print("Paragraph blocks:", len(paragraph_entries))
print("Table rows:", len(table_row_entries))
print()
for index, section_name in enumerate(full_report.keys(), start=1):
    print(f"{index:>2}. {section_name}")
    if index == 12:
        break


## Run the reviewer

This step runs `IUCNAssessmentReviewer.review_full_report(...)`, then applies `clean_up_violations(...)` so the displayed output is easier to inspect.


In [ ]:
project_modules = fresh_project_imports(verbose=False)
reviewer = project_modules["IUCNAssessmentReviewer"]()
violations = reviewer.review_full_report(full_report)
cleaned_violations = reviewer.clean_up_violations(list(violations))

configured_checkers = [checker.__class__.__name__ for checker in reviewer.checkers]
rule_class_counts = {}
for violation in cleaned_violations:
    rule_class_counts[violation.rule_class] = rule_class_counts.get(violation.rule_class, 0) + 1

print("Configured standard checkers:", configured_checkers)
print("Bibliography checker:", reviewer.bibliography_checker.__class__.__name__)
print("Violation count:", len(cleaned_violations))
print("Rule-class counts:")
for rule_class in sorted(rule_class_counts):
    print(f"- {rule_class}: {rule_class_counts[rule_class]}")


## Display violations

This final cell prints the cleaned violations as JSON-style dictionaries.


In [ ]:
print(json.dumps([violation.to_dict() for violation in cleaned_violations], indent=2, ensure_ascii=False))
